# Environment setup

In [ ]:
pip install torch torchvision wandb thop matplotlib

# Data & Custom Dataloader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import numpy as np
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [ ]:

# Determine device automatically
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1a. Compute mean and std for normalization
transform_temp = transforms.Compose([transforms.ToTensor()])
train_dataset_temp = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_temp)
loader_temp = DataLoader(train_dataset_temp, batch_size=1000, shuffle=False)

mean = 0.
std = 0.
total_images = 0
for images, _ in loader_temp:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images += batch_samples
mean /= total_images
std /= total_images

mean = mean.tolist()
std = std.tolist()
print("Calculated mean:", mean)
print("Calculated std:", std)

# 1b. Transform with normalization
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    # Padding & Cropping: The standard "trick" for CIFAR-10
    transforms.RandomCrop(32, padding=4),
    # Colour variation: Learning to ignore lighting differences
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    # Random Erasing: Advanced technique for robustness
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])


#
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


# Reload normalized datasets and prepare loaders
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

# Split the indices
train_indices, val_indices = random_split(range(len(full_train_dataset)), [train_size, val_size])

# Apply transformation on train and validation data
train_data = TransformedSubset(Subset(full_train_dataset, train_indices), transform=train_transform)
val_data = TransformedSubset(Subset(full_train_dataset, val_indices), transform=test_transform)


batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train, validation, and test loaders are ready with normalized CIFAR-10 data.")


Using device: cuda
Calculated mean: [0.49139973521232605, 0.48215848207473755, 0.4465309679508209]
Calculated std: [0.20230095088481903, 0.19941279292106628, 0.20096160471439362]
Train, validation, and test loaders are ready with normalized CIFAR-10 data.


# CNN-model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # Block 2
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        # Block 3
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # Fully Connected Head
        # After three 2x2 poolings, 32x32 becomes 4x4
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Block 2
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        # Block 3
        x = self.pool(F.relu(self.bn5(self.conv5(x))))

        # Classification
        x = x.view(-1, 128 * 4 * 4)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# FLOPs

# Training

In [ ]:
def train_or_validate(model, loader, criterion, optimizer=None):
    if optimizer:
        model.train()
    else:
        model.eval()
    loss_sum = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if optimizer:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if optimizer:
            loss.backward()
            optimizer.step()
        loss_sum += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    avg_loss = loss_sum / total
    accuracy = correct / total
    return avg_loss, accuracy


In [ ]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: m25csa012 (m25csa012-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Run experiment

In [ ]:
# model = CustomCNN()

from thop import profile
import torch.optim as optim

model = CustomCNN().to(device)
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []


# 1. Make sure input is on the same device as the model
input = torch.randn(1, 3, 32, 32).to(device)

# Note: make sure 'model' was already moved via model.to(device)
flops, params = profile(model, inputs=(input, ))

print(f"FLOPs: {flops}, Params: {params}")
# Log these to WandB


wandb.init(
    project="Jahanvi_Gajera_M25CSA012_lab2_worksheet",
    config={
        "epochs": 50,
        "batch_size": 128,
        "lr": 0.001,
        "architecture": "CustomCNN",
        "dataset": "CIFAR-10"
    }
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),wandb.config.lr)


for epoch in range(wandb.config.epochs):
    train_loss, train_acc = train_or_validate(model, train_loader, criterion, optimizer)
    val_loss, val_acc = train_or_validate(model, val_loader, criterion)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Train Acc={train_acc:.4f} | Val Loss={val_loss:.4f} Val Acc={val_acc:.4f}")


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 30675968.0, Params: 1194282.0


Epoch 1: Train Loss=1.7628 Train Acc=0.3516 | Val Loss=4.1262 Val Acc=0.2695
Epoch 2: Train Loss=1.4297 Train Acc=0.4840 | Val Loss=2.2930 Val Acc=0.3767
Epoch 3: Train Loss=1.2800 Train Acc=0.5412 | Val Loss=2.0902 Val Acc=0.4280
Epoch 4: Train Loss=1.1831 Train Acc=0.5778 | Val Loss=2.7263 Val Acc=0.3977
Epoch 5: Train Loss=1.1187 Train Acc=0.6034 | Val Loss=2.2579 Val Acc=0.4510
Epoch 6: Train Loss=1.0618 Train Acc=0.6260 | Val Loss=1.8892 Val Acc=0.5094
Epoch 7: Train Loss=1.0172 Train Acc=0.6427 | Val Loss=2.3405 Val Acc=0.4860
Epoch 8: Train Loss=0.9760 Train Acc=0.6568 | Val Loss=1.9524 Val Acc=0.5110
Epoch 9: Train Loss=0.9463 Train Acc=0.6685 | Val Loss=3.3180 Val Acc=0.4231
Epoch 10: Train Loss=0.9196 Train Acc=0.6757 | Val Loss=1.7400 Val Acc=0.5533
Epoch 11: Train Loss=0.8883 Train Acc=0.6895 | Val Loss=3.1896 Val Acc=0.4143
Epoch 12: Train Loss=0.8623 Train Acc=0.6998 | Val Loss=1.8610 Val Acc=0.5342
Epoch 13: Train Loss=0.8400 Train Acc=0.7055 | Val Loss=1.6131 Val Acc=0.

In [ ]:
wandb.log({
    "Observations": "Model starts overfitting after ~30 epochs. Training accuracy keeps improving while validation accuracy plateaus and fluctuates. Validation loss remains high, indicating poor generalization."
})
